<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/Image-to-Image%20Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Image-to-Image Translation with CycleGAN

CycleGAN allows for learning a mapping between an input image and an output image without the need for paired examples. This is perfect for transitions like Day → Night or Summer → Winter.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.utils import save_image
from PIL import Image
import os

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


### 1. Define the Architecture
CycleGAN uses two generators ($G: X \to Y$ and $F: Y \to X$) and two discriminators ($D_X$ and $D_Y$).

In [2]:
class ResidualBlock(nn.Module):
    def __init__(self, in_features):
        super(ResidualBlock, self).__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(in_features, in_features, 3),
            nn.InstanceNorm2d(in_features),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(in_features, in_features, 3),
            nn.InstanceNorm2d(in_features)
        )

    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, input_nc, output_nc, n_residual_blocks=9):
        super(Generator, self).__init__()
        # Initial convolution block
        model = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(input_nc, 64, 7),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True)
        ]
        # Downsampling
        in_features = 64
        out_features = in_features * 2
        for _ in range(2):
            model += [
                nn.Conv2d(in_features, out_features, 3, stride=2, padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features * 2
        # Residual blocks
        for _ in range(n_residual_blocks):
            model += [ResidualBlock(in_features)]
        # Upsampling
        out_features = in_features // 2
        for _ in range(2):
            model += [
                nn.ConvTranspose2d(in_features, out_features, 3, stride=2, padding=1, output_padding=1),
                nn.InstanceNorm2d(out_features),
                nn.ReLU(inplace=True)
            ]
            in_features = out_features
            out_features = in_features // 2
        # Output layer
        model += [nn.ReflectionPad2d(3), nn.Conv2d(64, output_nc, 7), nn.Tanh()]
        self.model = nn.Sequential(*model)

    def forward(self, x):
        return self.model(x)

### 2. Initialize Models and Optimizers
You would typically load your datasets (Domain A and Domain B) here before starting the training loop.

In [3]:
# Hyperparameters
lr = 0.0002
beta1 = 0.5
beta2 = 0.999

# Initialize generators and discriminators
netG_A2B = Generator(3, 3).to(device) # Day to Night
netG_B2A = Generator(3, 3).to(device) # Night to Day

# Loss functions
criterion_GAN = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_identity = nn.L1Loss()

# Optimizers
optimizer_G = optim.Adam(list(netG_A2B.parameters()) + list(netG_B2A.parameters()), lr=lr, betas=(beta1, beta2))

print("Models initialized. Ready for training loop implementation.")

Models initialized. Ready for training loop implementation.


In [4]:
class Discriminator(nn.Module):
    def __init__(self, input_nc):
        super(Discriminator, self).__init__()

        def discriminator_block(in_filters, out_filters, normalize=True):
            layers = [nn.Conv2d(in_filters, out_filters, 4, stride=2, padding=1)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_filters))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *discriminator_block(input_nc, 64, normalize=False),
            *discriminator_block(64, 128),
            *discriminator_block(128, 256),
            *discriminator_block(256, 512),
            nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(512, 1, 4, padding=1)
        )

    def forward(self, img):
        return self.model(img)

# Initialize discriminators
netD_A = Discriminator(3).to(device)  # Discriminator for Domain A
netD_B = Discriminator(3).to(device)  # Discriminator for Domain B

# Optimizers for discriminators
optimizer_D_A = optim.Adam(netD_A.parameters(), lr=lr, betas=(beta1, beta2))
optimizer_D_B = optim.Adam(netD_B.parameters(), lr=lr, betas=(beta1, beta2))

def train_step(real_A, real_B):
    # ------------------
    #  Train Generators
    # ------------------
    optimizer_G.zero_grad()

    # Identity loss
    loss_id_A = criterion_identity(netG_B2A(real_A), real_A)
    loss_id_B = criterion_identity(netG_A2B(real_B), real_B)
    loss_identity = (loss_id_A + loss_id_B) / 2

    # GAN loss
    fake_B = netG_A2B(real_A)
    loss_GAN_A2B = criterion_GAN(netD_B(fake_B), torch.ones_like(netD_B(fake_B)))
    fake_A = netG_B2A(real_B)
    loss_GAN_B2A = criterion_GAN(netD_A(fake_A), torch.ones_like(netD_A(fake_A)))
    loss_GAN = (loss_GAN_A2B + loss_GAN_B2A) / 2

    # Cycle loss
    recov_A = netG_B2A(fake_B)
    loss_cycle_A = criterion_cycle(recov_A, real_A)
    recov_B = netG_A2B(fake_A)
    loss_cycle_B = criterion_cycle(recov_B, real_B)
    loss_cycle = (loss_cycle_A + loss_cycle_B) / 2

    # Total G loss
    loss_G = loss_GAN + 10.0 * loss_cycle + 5.0 * loss_identity
    loss_G.backward()
    optimizer_G.step()

    # ---------------------
    #  Train Discriminators
    # ---------------------
    # D_A
    optimizer_D_A.zero_grad()
    loss_real_A = criterion_GAN(netD_A(real_A), torch.ones_like(netD_A(real_A)))
    loss_fake_A = criterion_GAN(netD_A(fake_A.detach()), torch.zeros_like(netD_A(fake_A.detach())))
    loss_D_A = (loss_real_A + loss_fake_A) / 2
    loss_D_A.backward()
    optimizer_D_A.step()

    # D_B
    optimizer_D_B.zero_grad()
    loss_real_B = criterion_GAN(netD_B(real_B), torch.ones_like(netD_B(real_B)))
    loss_fake_B = criterion_GAN(netD_B(fake_B.detach()), torch.zeros_like(netD_B(fake_B.detach())))
    loss_D_B = (loss_real_B + loss_fake_B) / 2
    loss_D_B.backward()
    optimizer_D_B.step()

    return loss_G.item(), loss_D_A.item(), loss_D_B.item()

print("Discriminators and training logic initialized.")

Discriminators and training logic initialized.


### 3. Training Loop Logic
In CycleGAN, we update the generators and discriminators alternately. The generator loss is a combination of:
- **Adversarial Loss**: To make the generated images look real.
- **Cycle Consistency Loss**: To ensure that $F(G(X)) \approx X$ and $G(F(Y)) \approx Y$.
- **Identity Loss**: To preserve color and composition when the generator is fed an image from the target domain.

In [5]:
def train_step(real_A, real_B):
    # ------------------
    #  Train Generators
    # ------------------
    optimizer_G.zero_grad()

    # Identity loss
    loss_id_A = criterion_identity(netG_B2A(real_A), real_A)
    loss_id_B = criterion_identity(netG_A2B(real_B), real_B)
    loss_identity = (loss_id_A + loss_id_B) / 2

    # GAN loss
    fake_B = netG_A2B(real_A)
    loss_GAN_A2B = criterion_GAN(netD_B(fake_B), torch.ones_like(netD_B(fake_B)))
    fake_A = netG_B2A(real_B)
    loss_GAN_B2A = criterion_GAN(netD_A(fake_A), torch.ones_like(netD_A(fake_A)))
    loss_GAN = (loss_GAN_A2B + loss_GAN_B2A) / 2

    # Cycle loss
    recov_A = netG_B2A(fake_B)
    loss_cycle_A = criterion_cycle(recov_A, real_A)
    recov_B = netG_A2B(fake_A)
    loss_cycle_B = criterion_cycle(recov_B, real_B)
    loss_cycle = (loss_cycle_A + loss_cycle_B) / 2

    # Total loss
    loss_G = loss_GAN + 10.0 * loss_cycle + 5.0 * loss_identity
    loss_G.backward()
    optimizer_G.step()

    # ---------------------
    #  Train Discriminators
    # ---------------------
    optimizer_D_A.zero_grad()
    loss_real = criterion_GAN(netD_A(real_A), torch.ones_like(netD_A(real_A)))
    loss_fake = criterion_GAN(netD_A(fake_A.detach()), torch.zeros_like(netD_A(fake_A.detach())))
    loss_D_A = (loss_real + loss_fake) / 2
    loss_D_A.backward()
    optimizer_D_A.step()

    optimizer_D_B.zero_grad()
    loss_real = criterion_GAN(netD_B(real_B), torch.ones_like(netD_B(real_B)))
    loss_fake = criterion_GAN(netD_B(fake_B.detach()), torch.zeros_like(netD_B(fake_B.detach())))
    loss_D_B = (loss_real + loss_fake) / 2
    loss_D_B.backward()
    optimizer_D_B.step()

    return loss_G.item(), loss_D_A.item(), loss_D_B.item()

print("Training step function defined.")

Training step function defined.
